In [1]:
# Import Libraries
import pandas as pd
from bs4 import BeautifulSoup
import requests
import smtplib
from datetime import time, datetime, timedelta
import random, re

# Global Variables
global bullet_characters 
global salary_descriptors 
bullet_characters = ['-', '•', '◦', '▪', '▸', '➤', '★', '*']
salary_descriptors = ['$','pay','paid','salary','compensation']

In [3]:
# Extract HTML - Paginated Func
def extract(page):    
    url = f'https://www.teamworkonline.com/jobs-in-sports?employment_opportunity_search%5Bsort_by%5D=most_recent&page={page}'
    user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36',
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"]
    
    headers = {'User-Agent': random.choice(user_agents_list)}

    page = requests.get(url,headers = headers)

    soup = BeautifulSoup(page.content,'html.parser')
    return(soup)

In [4]:
# Extract HTML - Inner Link (a tag) Func
def extract_inner(url):    
    user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36',
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"]
    
    headers = {'User-Agent': random.choice(user_agents_list)}

    page = requests.get(url,headers = headers)

    soup = BeautifulSoup(page.content,'html.parser')
    return(soup)

In [84]:
def transform(jobs_soup):
    all_jobs = jobs_soup.find_all(class_ = "browse-jobs-card")
    raw_jobs = []
    for job in all_jobs:
##################################################################### Scraping Basic info from Job cards ############################################################################
        try:
            raw_job={
                'logo_image': job.find('img').get('src') if job.find('img') else None,
                'job_url': 'https://www.teamworkonline.com' + job.find('a').get('href') if job.find('a') else None,
                'title': job.find(class_ = "margin-none").get_text().strip() if job.find(class_ = "margin-none") else None,
                'company': job.find(class_ = "browse-jobs-card__content--organization").get_text().strip() if job.find(class_ = "browse-jobs-card__content--organization") else None,
                'location_info': job.find(class_ = "trending__content--small").get_text().strip() if job.find(class_ = "trending__content--small") else None,
                'experience_level': job.find(class_ = "browse-jobs-card__content--small").get_text().strip() if job.find(class_ = "browse-jobs-card__content--small") else None  
            }
####################################################################### Extracting info from the individual job site #################################################################     
            job_soup = extract_inner(raw_job['job_url'])
            salary_fetcher(job_soup)
            
            
            
            
            
            
################################################### Scraping time based info from Job cards and doing minor extractions  #############################################################    
            posting_number_list = job.find_all(class_ = "browse-jobs-card__scoreboard")
            if len(posting_number_list)>= 2:
                raw_job['posting_numbers'] = posting_number_list[0].get_text().strip() + posting_number_list[1].get_text().strip()
            else:
                raw_job['posting_numbers'] = None
            raw_job['posting_date_identifier'] = job.find(class_ = "trending__content--small trending__scoreboard--time margin-left--xx-small").get_text().strip().split(' ')[0]
            raw_job['scrape_datetime'] = datetime.now()
            
            jobs_list.append(raw_job)
            
        except Exception as e:
            print(f"Error during job scrape: {e}")
            continue
    return

In [116]:
job_soup = extract_inner('https://www.teamworkonline.com/baseball-jobs/frontierleaguejobs/joliet-slammers/player-branding-and-development-coordinator-internship-2159945')

In [117]:
# This is code to find salary in the top portion of the sit
try:
    salary_flag = job_soup.find_all('div',class_ = 'margin-right--x-small')
    for i in range(0,len(salary_flag)):
        if any(descriptor in str(salary_flag[i]) for descriptor in salary_descriptors):
            #salary_desc = salary_flag[i].get_text()
            print(salary_flag[i].get_text())
        else:
            pass
except Exception as e:
    print(f"Error during job scrape: {e}")

In [125]:
##### START HERE TODAY

job_inner = job_soup.find('div', class_='opportunity-preview__body')
job_details = job_inner.find_all('p')
if job_details:
    print('Gotcha P')
    if job_inner.find('ul'):
        p_tag = True
    else:
        p_tag = False
else:
    print('Nope Div')
    p_tag = False
    job_details = job_soup.find('div', class_='opportunity-preview__body').find_all('div')

#.find_all(['p','strong']) •
data = {}
headers = []
for section in job_details:
    for br in section.find_all('br'):
        br.replace_with("\n")
   
    section = wrap_bullets(section.get_text())
    #print(section)
    #print('\n\n\n')

Nope Div
<p>Essential Duties &amp; Responsibilities:</p>
<p>Essential Duties &amp; Responsibilities:</p>


In [139]:
############# THIS IS THE NEW APPROACH AS OF 3/17. IT NEEDS TO BE WORKED ON AS I HAVENT reintegreated this properly. #######
###############  Its working better but I need it to get all p tags from wrap_bullets func not just 1 ################# 

#text = "This is a sample test to see what will happen to be honest: \n\n\n- This is bullet 1.\n\n\n\n- This is bullet 2.\n- This is bullet 3.\n- This is bullet 4.\n- This is bullet 5. \n This is extra text Now that Idk what will happen to."

#wrap_bullets(text)

def wrap_bullets(text):
    
    lines = text.split('\n')
    result = []
    in_list = False
    
    for line in lines:
        stripped_line = line.strip()
        
        if in_list and stripped_line == '':
            continue
    
        if stripped_line and stripped_line[0] in bullet_characters:
            if not in_list:
                preceding_line = next((line for line in reversed(result) if line.strip()), None)
                result.append(f'<p>{preceding_line}</p>')
                result.append('<ul>')
                in_list = True
            content = stripped_line[1:].strip()
            result.append(f'<li>{content}</li>')
        else:
            if in_list:
                result.append('</ul>')
                in_list = False
            result.append(stripped_line)
            
    if in_list:
        result.append('</ul>')
    
    final_result = '\n'.join(result)
    
    soup = BeautifulSoup(final_result,'html.parser')
    print(soup)
    
    return(soup.find('p'))

In [120]:
job_inner = job_soup.find('div', class_='opportunity-preview__body')
job_details = job_inner.find_all('p')
if job_details:
    print('Gotcha P')
    if job_inner.find('ul'):
        p_tag = True
    else:
        p_tag = False
else:
    print('Nope Div')
    p_tag = False
    job_details = job_soup.find('div', class_='opportunity-preview__body').find_all('div')

print(p_tag)
print(job_details)

Nope Div
False
[<div data-controller="blank-link-target"><div>Player Branding &amp; Development Program Coordinator – Internship<br/><br/>Department: Baseball Operations / Media<br/>Location: Joliet, IL<br/>Reports To: Field Manager &amp; Director of Baseball Operations<br/>Status: Internship (Academic Credit)<br/><br/>About the Joliet Slammers:<br/><br/>The Joliet Slammers are Joliet’s hometown professional baseball team, competing in the Frontier League — a Partner League of Major League Baseball. We’re dedicated to delivering unforgettable experiences for fans throughout the city of Joliet and the greater Chicagoland area. We pride ourselves on a positive, team-first culture rooted in community, fun, and creating memories for fans of every age and we’re looking for the right teammate to help us keep the momentum going.<br/><br/>Program Overview<br/><br/>The Player Branding &amp; Development Program is a season-long initiative designed to help Slammers players build their personal br

In [142]:
#.find_all(['p','strong']) •
data = {}
headers = []
for section in job_details:
    if (p_tag):
        full_text = section
        
        if section.find('strong'):
            header = section.find('strong').get_text()
            content = section.find_next_sibling()
        else:
            header = ''
            content = ''
    else:
        full_text = wrap_bullets(section.get_text())
        
    #for br in section.find_all('br'):
    #    br.replace_with("\n")
    
    #full_text = section.get_text(separator = '\n')
    if not full_text:
        continue
    
    #print(full_text)
    #header_text,details = extract_bullets_from_text(full_text,header,content)
    
    #if header_text and details:
    #    headers.append(header_text)
    #    data[header_text] = details
    #else:
    details = []
    header_text = full_text.get_text()
    next_tag = full_text.find_next_sibling()
    
    if (next_tag and next_tag.name in ['ul', 'ol']):
        headers.append(header_text)
        items = next_tag.find_all('li')
        details = [item.get_text(strip=True) for item in items]
    
        data[header_text] = details 
            

# Convert to single-row DataFrame
df = pd.DataFrame({k: pd.Series(v) for k, v in data.items()})
print(df)

Player Branding &amp; Development Program Coordinator – Internship

Department: Baseball Operations / Media
Location: Joliet, IL
Reports To: Field Manager &amp; Director of Baseball Operations
Status: Internship (Academic Credit)

About the Joliet Slammers:

The Joliet Slammers are Joliet’s hometown professional baseball team, competing in the Frontier League — a Partner League of Major League Baseball. We’re dedicated to delivering unforgettable experiences for fans throughout the city of Joliet and the greater Chicagoland area. We pride ourselves on a positive, team-first culture rooted in community, fun, and creating memories for fans of every age and we’re looking for the right teammate to help us keep the momentum going.

Program Overview

The Player Branding &amp; Development Program is a season-long initiative designed to help Slammers players build their personal brands, grow their fan engagement, and prepare for opportunities beyond the playing field. Through a blend of media 

In [52]:
def extract_bullets_from_text(text,strong_header,strong_content):
    """Extract header and bullets from text"""
    print(text)
    print('\n\n\n')
    try:
        if strong_content and strong_header:
            header = strong_header
            content = strong_content.get_text().strip()
            print(header)
        else:
            if ':' in text:
                sections = text.split(':')
                #print(sections)
                header = sections[0].strip()
                content = sections[1:]
                #if sections[1].strip() != '' else header
                #print(content)
            elif '\n' in text:
                sections = text.split('\n', 1)
                #print(sections)
                #print('\n\n')
                header = sections[0].strip()
                content = sections[1].strip() if sections[1].strip() != '' else header
                
                
        if header and content:
            bullets = re.split(r'[-•·*]\s*', content)
            
            if len(bullets) <= 1:
                bullets = re.split(r'-\s+', content)
            
            bullets = [b.strip() for b in bullets if b.strip()]
            
            if bullets:
                return (header, bullets)
        
    except:
        print('Error')
        pass
    
    return (None, None)

In [138]:
headers

['Essential Duties & Responsibilities:',
 'Essential Duties & Responsibilities:']

In [71]:
# CODE TO PULL OFF RELEVANT INFO FROM DETAIL SCRAPE
df.filter(regex = '(?i)requ|respon|qual|look|skill|know|job|function',axis=1)

""
0


In [83]:
details_df = []
for header in headers:
    details_df.append({header: df[header].str.split('\n').explode(header)})
details_df = pd.DataFrame(details_df)
details_df

,Player Branding & Development Program Coordinator – InternshipDepartment
0,0 Baseball Operations / MediaLocation Name:...
1,0 Baseball Operations / MediaLocation Name:...


In [80]:
# Will use this once deployed pages = (1,11)
pages = ((1,3))
jobs_list = []
for page in range(pages[0],pages[1]):
    recent_jobs_html = extract(page)
    transform(recent_jobs_html)

In [97]:
team_work_df = pd.DataFrame(jobs_list)

team_work_df['city'] = team_work_df['location_info'].str.split('·').str[0].str.strip()
team_work_df['state'] = team_work_df['location_info'].str.split('·').str[1].str.strip()

In [110]:
team_work_df.loc[team_work_df['state'].str.len()>2,['title','location_info','city']]

,title,location_info,city
91,HR Business Partner,London · United Kingdom · Hybrid,London
98,"Major Gift Officer, West Coast",CA · Remote,CA
131,Event Operations + Merchandising Intern,FL · Hybrid,FL
132,Event Merchandising Manager (On-Site),FL · Hybrid,FL
134,Creative Strategist,United States · Remote,United States
135,Account Executive,United States · Remote,United States
231,Event Operations + Merchandising Intern,FL · Hybrid,FL
232,Event Merchandising Manager (On-Site),FL · Hybrid,FL
234,Creative Strategist,United States · Remote,United States
235,Account Executive,United States · Remote,United States


In [25]:
# Will use this once deployed pages = ((1,3),(3,5),(5,7),(7,9),(9,11))
pages = ((1,3))
for page_range in pages:
    for page_ind in range(page_range[0],page_range[1]):
        recent_jobs_html = extract(page_ind)
        transorm(recent_jobs_html)

1
2
3
4
5
6
7
8
9
10


In [12]:
### This is a copy of scaper code so I can split up some processing

all_jobs = soup.find_all(class_ = "browse-jobs-card")

for job in all_jobs:
##################################################################### Scraping Basic info from Job cards ####################################################
    
    logo_image = job.find('img').get('src')
    job_url = 'https://www.teamworkonline.com' + job.find('a').get('href')
    title = job.find(class_ = "margin-none").get_text().strip()
    company = job.find(class_ = "browse-jobs-card__content--organization").get_text().strip()
    location_info = job.find(class_ = "trending__content--small").get_text().strip()
    # Will need to split as such location_info.split('·')[0])
    experience = job.find(class_ = "browse-jobs-card__content--small").get_text().strip()
    
################################################### Scraping time based info from Job cards and doing minor calcualtions ####################################
    posting_number_list = job.find_all(class_ = "browse-jobs-card__scoreboard")
    posting_numbers = posting_number_list[0].get_text().strip() + posting_number_list[1].get_text().strip()
    posting_date_identifier = job.find(class_ = "trending__content--small trending__scoreboard--time margin-left--xx-small").get_text().strip().split(' ')[0]
    if posting_date_identifier.endswith('s'):
        pass
    else:
        posting_date_identifier = posting_date_identifier + ('s')
    posting_timestamp = datetime.now() - timedelta(**{posting_date_identifier: int(posting_numbers)})

'02/27/2026 12:58:28'

tests
